In [114]:
from collections import defaultdict
import os
import json


In [115]:
class GraphTraversal:
	def __init__(self, base_dir: str):
		self.base_dir = base_dir
		self.gameplay_options_count = {}

		self.visited = set()
	
	def build_full_id(self, filename: str, object_names: list[str], node_id: str):
		return ".".join([filename] + object_names + [node_id])
	
	def extract_next_nodes(self, node: dict, full_id: str):
		next_nodes = []
		node_type = node.get("type")

		if "next" in node:
			next_nodes.append(node["next"])

		elif node_type == "choice" and "choices" in node:
			self.gameplay_options_count[full_id] = len(node["choices"])

			for choice in node["choices"]:
				if "next" in choice:
					next_nodes.append(choice["next"])

		elif node_type == "condition" and "conditions" in node:
			for cond in node["conditions"]:
				if "next" in cond:
					next_nodes.append(cond["next"])

		return next_nodes
	
	def dfs_traverse(self, filename: str, object_names: list[str], node_id: str, node_map: dict):
		full_id = self.build_full_id(filename, object_names, node_id)

		if full_id in self.visited:
			return

		self.visited.add(full_id)

		node = node_map.get(node_id)
		if node:			
			next_nodes = self.extract_next_nodes(node, full_id)

			for next_node in next_nodes:
				self.dfs_traverse(filename, object_names, next_node, node_map)

	def traverse_graph(self, filename: str, object_names: list[str], node_map: dict):
		if "root" in node_map:
			self.dfs_traverse(filename, object_names, "root", node_map)
		else:
			for sub_name, sub_map in node_map.items():
				self.traverse_graph(filename, object_names + [sub_name], sub_map)

	def strip_base_and_ext(self, full_path: str, base_dir: str):
		full_path = os.path.normpath(full_path)
		base_dir = os.path.normpath(base_dir)

		rel_path = os.path.relpath(full_path, base_dir)
		rel_path = os.path.splitext(rel_path)[0]

		return rel_path.replace(os.sep, "/")
	
	def run(self):
		# Genera los nombres de archivos y directorios de todos los niveles en un arbol de directorios
		for root, _, files in os.walk(self.base_dir):
			dir_name = os.path.basename(root).lower()
			if dir_name != "computer":
				for file in files:
					if file.endswith(".json") and file.lower() != "test.json":
						full_path = os.path.join(root, file)
						
						filename = os.path.splitext(os.path.basename(full_path))[0]
						filename = filename[:1].upper() + filename[1:]

						with open(full_path, "r", encoding="utf-8") as f:
							data = json.load(f)

						self.traverse_graph(filename, [], data)

		print(f"Total visited nodes: {len(self.visited)}")


In [116]:
localization_dir = "../localization"
structure_dir = os.path.join(localization_dir, "structure")


In [117]:
graph = GraphTraversal(structure_dir)
graph.run()


Total visited nodes: 661


In [118]:
print(len(graph.gameplay_options_count))
print(graph.gameplay_options_count)


47
{'GeneralDialogs.bed.goToSleep': 2, 'Scene1Bedroom1.computer1.choices': 2, 'Scene1Bedroom1.computer1.choices2': 2, 'Scene1Bedroom1.computer2.root': 2, 'Scene1Bedroom1.computer2.choices2': 4, 'Scene1Bedroom1.computer2.choices3': 2, 'Scene1Bedroom2.computer.choices': 2, 'Scene1Bedroom2.computer.choices2': 2, 'Scene1Bedroom2.computer.choices3': 2, 'Scene1Classroom.part2.thanks': 3, 'Scene1Lunch1.main.choices': 2, 'Scene1Lunch1.phone.choices': 1, 'Scene2Bedroom.computer.reply': 1, 'Scene2Bedroom.phone.choices': 2, 'Scene2Break.part2.choice': 3, 'Scene3Bedroom.main.choices': 2, 'Scene3Bedroom.main.choices2': 2, 'Scene3Break.conversation1.choices': 2, 'Scene3Break.conversation1.choices2': 2, 'Scene3Break.conversation2.choices3': 2, 'Scene4Backyard.mainConversation.choices': 2, 'Scene4Bedroom.phone.root': 1, 'Scene4Bedroom.phone.choices': 3, 'Scene4Bedroom.phone.choices2': 2, 'Scene4Garage.phone1.root': 1, 'Scene4Garage.photo.choices': 2, 'Scene4Garage.phone2.choices': 1, 'Scene4Garage.int

In [119]:
data_dir = "./data"
raw_dir = os.path.join(data_dir, "processed")
session_dir = os.path.join(raw_dir, "merged")
gameplay_dir = os.path.join(session_dir, "Gameplay")
traces_path = os.path.join(gameplay_dir, "traces.ndjson")


In [120]:
traces = []
invalid_count = 0

with open(traces_path, "r", encoding="utf-8") as f:
	for line in f:
		line = line.strip()
		if line:
			try:
				traces.append(json.loads(line))
			except json.JSONDecodeError:
				invalid_count += 1

print("Total valid traces:", len(traces))
print("Total invalid traces:", invalid_count)


Total valid traces: 98930
Total invalid traces: 0


In [121]:
OPTION_SELECT_OBJ_ID = "https://w3id.org/xapi/seriousgames/activity-types/dialog-tree/OptionSelect"
NODE_EXT_ID = "https://w3id.org/xapi/seriousgame/extensions/Node"
RESPONSE_EXT_ID = "https://w3id.org/xapi/seriousgame/extensions/Response"


In [122]:
user_options = defaultdict(set)

for trace in traces:
	obj_id = trace.get("object", {}).get("id", "")

	if obj_id == OPTION_SELECT_OBJ_ID:
		result = trace.get("result", {})
		extensions = result.get("extensions", {}) or {}

		node = extensions.get(NODE_EXT_ID)
		response = extensions.get(RESPONSE_EXT_ID)

		user_options[node].add(response)

user_options_count = {}

for node, responses in user_options.items():
	user_options_count[node] = len(responses)


In [123]:
print(user_options)


defaultdict(<class 'set'>, {'Scene1Classroom.part2.thanks': {'... Ah, sí, hola.', 'Igualmente, encantado de conocerte *sonríes*.', 'Igualmente, encantada de conocerte *sonríes*.', 'Gracias, si necesito algo ya te iré diciendo.'}, 'Scene1Lunch1.main.choices': {'Contarles sobre mi día.', 'No contarles sobre mi día.'}, 'Scene1Lunch1.phone.choices': {'Mandar la foto de Toni y Mila.'}, 'Scene1Bedroom1.computer1.choices': {'Contestar más friamente.', 'Contestar amablemente.'}, 'Scene1Bedroom1.computer1.choices2': {'Su perrita.', 'Su cercanía.'}, 'Scene1Bedroom1.computer2.root': {'Hace poco.', 'Un par de años.'}, 'Scene1Bedroom1.computer2.choices2': {'1º de la ESO', '2º de la ESO', '3º de la ESO', '4º de la ESO'}, 'Scene1Bedroom1.computer2.choices3': {'Me voy ya, que me van a echar la bronca', 'Me voy ya, que es tarde'}, 'Scene1Bedroom1.bed.goToSleep': {'Sí.', 'No.'}, 'Scene1Bedroom2.computer.choices': {'No contarle mi día.', 'Contarle mi día.'}, 'Scene1Bedroom2.computer.choices2': {'¿Estás b

In [124]:
missing_keys = user_options_count.keys() - graph.gameplay_options_count.keys()
print(missing_keys)


{'Scene1Bedroom2.bed.goToSleep', 'Scene5Bedroom.bed.goToSleep', 'Scene2Bedroom.bed.goToSleep', 'Scene6BedroomRouteA1.bed.goToSleep', 'Scene1Bedroom1.bed.goToSleep', 'Scene6Bedroom.bed.goToSleep', 'Scene4Bedroom.bed.goToSleep'}


In [125]:
for key in missing_keys:
	normalized_key = key.split(".", 1)[1]
	user_options_count[normalized_key] = user_options_count.pop(key)

print(user_options_count)


{'Scene1Classroom.part2.thanks': 4, 'Scene1Lunch1.main.choices': 2, 'Scene1Lunch1.phone.choices': 1, 'Scene1Bedroom1.computer1.choices': 2, 'Scene1Bedroom1.computer1.choices2': 2, 'Scene1Bedroom1.computer2.root': 2, 'Scene1Bedroom1.computer2.choices2': 4, 'Scene1Bedroom1.computer2.choices3': 2, 'Scene1Bedroom2.computer.choices': 2, 'Scene1Bedroom2.computer.choices2': 2, 'Scene1Bedroom2.computer.choices3': 2, 'Scene2Break.part2.choice': 3, 'Scene2Bedroom.computer.reply': 1, 'Scene2Bedroom.phone.choices': 2, 'Scene3Break.conversation1.choices': 2, 'Scene3Break.conversation2.choices3': 2, 'Scene3Bedroom.main.choices': 2, 'Scene3Bedroom.main.choices2': 2, 'Scene4Backyard.mainConversation.choices': 2, 'Scene4Garage.phone1.root': 1, 'Scene4Garage.photo.choices': 2, 'Scene4Garage.phone2.choices': 1, 'Scene4Garage.interruption.choices': 2, 'Scene4Bedroom.phone.root': 1, 'Scene4Bedroom.phone.choices': 3, 'Scene4Bedroom.phone.choices2': 2, 'Scene5Livingroom.choices': 2, 'Scene3Break.conversation

In [126]:
total_gameplay = 0

for node, gameplay_count in graph.gameplay_options_count.items():
	user_count = user_options_count.get(node)

	if user_count is None:
		normalized_node = node.split(".", 1)[1]
		user_count = user_options_count.get(normalized_node, 0)

	if user_count < gameplay_count:
		print(f"{node}: {user_count}/{gameplay_count} options explored")
	else:
		total_gameplay += 1

percentage = total_gameplay / len(graph.gameplay_options_count) * 100

print(f"Nodes with all options explored: {(percentage):.2f}%")


Nodes with all options explored: 100.00%
